# Locomotion State (improved) — Kinematic Axis (HSI 1.3)

Deterministic rule producing the `locomotion_state` categorical reading for the `kinematic` domain (RFC-HSI-0010): is the body moving through space, and how — **stationary / on_foot / in_vehicle**.

## Overview
- Objective: fix the worst confusion in the mentor pipeline — **stationary <-> in_vehicle** (~5000 windows leak across that boundary). Both are low body-motion, so a movement-only rule cannot separate a still body from a still (or smoothly riding) vehicle. This axis separates them with **non-motion signals** instead.
- Why this is hard: an accelerometer sees *body motion*, not displacement. A stopped car and a standing person both read "barely moving", so movement alone collapses them together. The original `locomotion_state` rule labelled "moving but not stepping" as in_vehicle, which leaves a stopped or smooth-riding vehicle indistinguishable from stationary.
- Approach (GPS-free, deterministic):
  - `on_foot`: rhythmic stepping — a high autocorrelation peak at step cadence (the rhythm feature reused from the other axes). Unchanged; this part already works.
  - `stationary` vs `in_vehicle`: decided by the magnetometer (de-rotated inclination / declination shift + field variance) and a corroborating engine-idle FFT band, not by movement.
- Core signal: a still body outdoors matches the local geomagnetic baseline; a still body inside a metal vehicle sits in a field distorted by the ferrous chassis — shifted inclination/declination, offset magnitude, higher variance. That distortion is the deterministic vehicle signature, and it is present even at a standstill.
- Sensors: accelerometer + magnetometer (required); gyroscope, barometer, orientation quaternion (optional, improve the rule). **GPS is not used** — the whole point is GPS-free separation.
- Sampling: 100 Hz (SHL; the 10-25 Hz engine-idle band needs >= 50 Hz). Window: 5 s (500 samples), consistent with the other axes.
- Classifier: deterministic rule (fixed thresholds; no training). All constants live in one config cell.
- Scope of this notebook (loop 1): build the feature extractor and rule, and verify on **synthetic** windows that the features compute and the thresholds separate the constructed cases. **No real-data validation and no accuracy claims** — SHL Hips/pocket and own-phone CSV are the next loop. Thresholds are **provisional**.

## 1. Setup and configuration
Imports and all constants in one place. Every threshold is **provisional** — tuned to separate the synthetic constructions below, pending real SHL / phone data.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from sklearn.metrics import confusion_matrix

mpl.rcParams.update({
    "figure.figsize": (7.5, 5.0),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "font.size": 11,
    "legend.fontsize": 10,
})

In [ ]:
# Sampling and windowing
FS         = 100                 # sample rate (Hz). SHL is 100 Hz; the 10-25 Hz idle band needs FS >= 50.
WINDOW_SEC = 5.0
WINDOW     = int(WINDOW_SEC * FS)  # 500 samples, consistent with the other axes

# on_foot (rhythm) gate
STEP_T     = 0.30               # autocorrelation peak above this = repeating stepping
MOVE_FLOOR = 0.30               # std of |accel| floor; below it the body is not moving enough to be on_foot
                                # (also stops a strong periodic engine tone from being read as stepping)
STEP_LAG   = (15, 128)          # autocorr lag band ~0.78-6.7 Hz cadence at 100 Hz (same as the other axes)

# Local geomagnetic baseline = the "outdoor still-body" reference. Source either the WMM / IGRF field
# model for the deployment site, or a recent stationary-outdoor self-calibration window. Mid-latitude
# defaults below. NOTE: the magnitude/angle-deviation cues need this baseline; field variance and the
# engine-idle FFT do not (they are absolute), so they survive when the baseline is unknown.
FIELD_REF_UT = 48.0            # local total field intensity (uT)
INC_REF_DEG  = 62.0            # local inclination / magnetic dip (deg)
DEC_REF_DEG  = 0.0             # local declination (deg)

# stationary-vs-vehicle thresholds (the hard split)
MAG_DEV_T   = 8.0             # |field magnitude - baseline| (uT) above this  -> distorted field
INC_DEV_T   = 10.0           # |inclination - baseline| (deg) above this      -> distorted field
DEC_DEV_T   = 15.0           # |declination - baseline| (deg) above this      -> distorted field
FIELD_VAR_T = 2.0           # std of field magnitude within window (uT) above this -> unstable field
IDLE_T      = 2.0          # in-band spectral density / whole-spectrum density above this -> engine tone
IDLE_BAND   = (10.0, 25.0)  # engine-idle vibration band (Hz)

CLASSES = ["stationary", "on_foot", "in_vehicle"]
CLASS_COLOR = {"stationary": "#55A868", "on_foot": "#4C72B0", "in_vehicle": "#DD8452"}

## 2. Feature extractor (deterministic)
One window of `(accel, mag, [gyro], [baro], [quat])` -> interpretable features. Two helpers first (quaternion rotation matrix; smallest angle difference), then the four feature groups:

- **movement** = std of |accel| (orientation-invariant) — how much the body moves.
- **step_rhythm** = autocorrelation peak in the cadence lag band — does the motion repeat (stepping)?
- **idle_band_ratio** = in-band spectral density vs the whole-spectrum density — a narrowband engine tone in 10-25 Hz reads >> 1, while broadband sensor noise or road vibration reads ~1. Needs FS >= 50 Hz.
- **magnetic_features** = field magnitude and within-window variance always; **inclination** once a down reference is known (gravity or quaternion); **declination** only with a full orientation quaternion. The `mode` flag records how far the de-rotation got, so the rule can degrade gracefully.

The magnetometer is the centrepiece. Raw magnitude `sqrt(Bx^2+By^2+Bz^2)` is rotation-invariant but throws away direction. De-rotating the field into global North-East-Down recovers the **inclination** (dip below horizontal) and **declination** (horizontal bearing), which is where the vehicle distortion shows up. With a quaternion the de-rotation is exact; with gravity alone the dip is still recoverable (the angle of the field relative to the vertical), but declination is not (it needs a true-north reference) and is returned as NaN.

In [ ]:
def quat_to_R(q):
    """Rotation matrix for quaternion q=[w,x,y,z] (device -> world / NED)."""
    w, x, y, z = q
    return np.array([
        [1 - 2*(y*y + z*z), 2*(x*y - w*z),     2*(x*z + w*y)],
        [2*(x*y + w*z),     1 - 2*(x*x + z*z), 2*(y*z - w*x)],
        [2*(x*z - w*y),     2*(y*z + w*x),     1 - 2*(x*x + y*y)],
    ])

def ang_diff(a, b):
    """Smallest absolute difference between two angles (deg)."""
    return abs((a - b + 180.0) % 360.0 - 180.0)


def movement(accel):
    """std of |accel| (m/s^2) — orientation-invariant body-motion amount."""
    return float(np.sqrt((accel**2).sum(axis=1)).std())

def step_rhythm(accel):
    """Autocorrelation peak of |accel| in the cadence lag band; high = repeating stepping."""
    a = np.sqrt((accel**2).sum(axis=1)); a = a - a.mean()
    ac = np.correlate(a, a, mode="full")[len(a) - 1:]
    lo, hi = STEP_LAG
    return 0.0 if ac[0] == 0 else float((ac / ac[0])[lo:hi].max())

def idle_band_ratio(accel, fs=FS):
    """In-band spectral density / whole-spectrum density. ~1 for broadband motion (sensor noise,
    road vibration); >> 1 only when a narrowband engine tone sits in 10-25 Hz. Needs fs >= 2*hi."""
    if fs < 2 * IDLE_BAND[1]:
        return np.nan                                       # band above Nyquist: feature unavailable
    a = np.sqrt((accel**2).sum(axis=1)); a = a - a.mean()
    spec = np.abs(np.fft.rfft(a * np.hanning(len(a))))**2
    freqs = np.fft.rfftfreq(len(a), 1.0 / fs)
    band = (freqs >= IDLE_BAND[0]) & (freqs <= IDLE_BAND[1])
    ref = freqs > 0.5                                       # whole usable spectrum = broadband floor
    return float(spec[band].mean() / (spec[ref].mean() + 1e-12))

def magnetic_features(mag, quat=None, gravity=None):
    """Field magnitude/variance always; inclination if a down reference (gravity or quat) is given;
    declination only with a quaternion. 'mode' flags how far the de-rotation got."""
    m = np.sqrt((mag**2).sum(axis=1))
    out = {"field_mag": float(m.mean()), "field_var": float(m.std()),
           "inclination": np.nan, "declination": np.nan, "mode": "magnitude_only"}
    if quat is not None:                                    # full North-East-Down de-rotation
        B = np.array([quat_to_R(q) @ b for q, b in zip(quat, mag)])
        north, east, down = B.mean(axis=0)
        out["inclination"] = float(np.degrees(np.arctan2(down, np.hypot(north, east))))
        out["declination"] = float(np.degrees(np.arctan2(east, north)))
        out["mode"] = "full_ned"
    elif gravity is not None:                               # gravity-only: inclination yes, declination no
        down_hat = -gravity / (np.linalg.norm(gravity) + 1e-9)   # accel reads "up"; down = -up
        Bm = mag.mean(axis=0)
        b_vert = Bm @ down_hat
        b_horiz = np.linalg.norm(Bm - b_vert * down_hat)
        out["inclination"] = float(np.degrees(np.arctan2(b_vert, b_horiz)))
        out["mode"] = "gravity_only"                        # declination NEEDS_QUAT -> NaN
    return out

## 3. Decision rule
Two branches. First the rhythm gate: a clear repeating cadence with enough body motion is `on_foot`. Everything else — low or non-rhythmic body motion, where movement cannot tell a still body from a still/smooth vehicle — goes to the stationary-vs-vehicle split, decided entirely by the non-motion cues.

A window is `in_vehicle` if **any** distortion cue fires: a magnitude / inclination / declination shift away from the geomagnetic baseline, an unstable field (high within-window variance), or an engine tone in the idle band. Cues that need a feature the data cannot supply (declination without a quaternion, the idle band below 50 Hz) are simply skipped — the rule uses whatever is available.

In [ ]:
def locomotion(win, fs=FS, ref=None):
    """win = dict(accel, mag, [gyro], [baro], [quat]). Returns (label, feats).
    ref = geomagnetic baseline; defaults to the config constants."""
    ref = ref or {"field_mag": FIELD_REF_UT, "inclination": INC_REF_DEG, "declination": DEC_REF_DEG}
    accel, mag, quat = win["accel"], win["mag"], win.get("quat")

    mv = movement(accel)
    rhythm = step_rhythm(accel)
    if rhythm > STEP_T and mv > MOVE_FLOOR:
        return "on_foot", {"movement": mv, "rhythm": rhythm}

    # low or non-rhythmic body motion: stationary vs in_vehicle from NON-motion signals only
    mf = magnetic_features(mag, quat=quat, gravity=accel.mean(axis=0))
    idle = idle_band_ratio(accel, fs)

    cues = {
        "mag_dev":   abs(mf["field_mag"] - ref["field_mag"]) > MAG_DEV_T,
        "inc_dev":   (not np.isnan(mf["inclination"])) and abs(mf["inclination"] - ref["inclination"]) > INC_DEV_T,
        "dec_dev":   (not np.isnan(mf["declination"])) and ang_diff(mf["declination"], ref["declination"]) > DEC_DEV_T,
        "field_var": mf["field_var"] > FIELD_VAR_T,
        "idle":      (not np.isnan(idle)) and idle > IDLE_T,
    }
    label = "in_vehicle" if any(cues.values()) else "stationary"
    return label, {"movement": mv, "rhythm": rhythm, "idle": idle, **mf, "cues": cues}

## 4. Data interface (loader stub)
`load_windows(source)` yields windows as `dict(accel, gyro, mag, baro, quat, label)`, so a real source plugs in next loop without touching the extractor or rule:

- **SHL** (Sussex-Huawei Locomotion, 100 Hz, Hand/Hips/Bag/Torso): `*_Motion.txt` carries accel / gyro / magnetometer / orientation quaternion; pressure gives the barometer; `Label.txt` maps to classes (`Still -> stationary`, `Walk/Run -> on_foot`, `Car/Bus/Train/Subway -> in_vehicle`). SHL supplies exactly the modalities this rule wants and is the GPS-free transport-mode benchmark — the intended validation set.
- **own phone CSV** (e.g. phyphox): columns `ax,ay,az,gx,gy,gz,mx,my,mz,pressure`, plus a quaternion if the orientation sensor is logged; degrade to gravity-only or magnitude-only otherwise.

This loop returns synthetic windows so the extractor and rule can be exercised end to end; the real branches raise `NotImplementedError` with the interface they must satisfy.

In [ ]:
def load_windows(source="synthetic", n_per=200, seed=0):
    """Yield dict(accel, gyro, mag, baro, quat, label). Real sources plug in next loop."""
    if source == "synthetic":
        return list(_synthetic_windows(n_per=n_per, seed=seed))
    if source in ("shl", "phone"):
        raise NotImplementedError(
            f"loop 2: read {source} into windows of "
            "dict(accel=(N,3), gyro=(N,3), mag=(N,3), baro=(N,) or None, quat=(N,4) or None, label=str); "
            "map labels to ['stationary','on_foot','in_vehicle'].")
    raise ValueError(source)


# --- synthetic windows: still body / parked-idling vehicle / driving vehicle / walking ---
G = 9.81
SYN_TRUE = {"stationary": "stationary", "vehicle_idle": "in_vehicle",
            "vehicle_moving": "in_vehicle", "walking": "on_foot"}

def _ned_ref_vec(total, inc, dec):
    inc, dec = np.radians(inc), np.radians(dec)
    return np.array([total*np.cos(inc)*np.cos(dec), total*np.cos(inc)*np.sin(dec), total*np.sin(inc)])

def _synthetic_windows(n_per=200, seed=0):
    rng = np.random.default_rng(seed)
    def random_quat():
        v = rng.normal(size=4); v /= np.linalg.norm(v)
        return -v if v[0] < 0 else v
    B_ned = _ned_ref_vec(FIELD_REF_UT, INC_REF_DEG, DEC_REF_DEG)   # outdoor baseline
    t = np.arange(WINDOW) / FS
    for kind in SYN_TRUE:
        for _ in range(n_per):
            q = random_quat(); R = quat_to_R(q)            # fixed device orientation over the window
            grav_dev = R.T @ np.array([0.0, 0.0, -G])      # NED: "up" reaction is negative-down
            if kind == "stationary":                       # still body, undistorted field
                accel = grav_dev + rng.normal(0, 0.03, (WINDOW, 3))
                mag = (R.T @ B_ned) + rng.normal(0, 0.3, (WINDOW, 3))
            elif kind == "vehicle_idle":                   # parked, engine running -> 18 Hz tone + distortion
                idle = 0.25 * np.sin(2*np.pi*18*t)
                accel = grav_dev + np.c_[idle, 0.5*idle, idle] + rng.normal(0, 0.05, (WINDOW, 3))
                mag = R.T @ _ned_ref_vec(FIELD_REF_UT+14, INC_REF_DEG-20, DEC_REF_DEG+25) + rng.normal(0, 2.5, (WINDOW, 3))
            elif kind == "vehicle_moving":                 # driving -> broadband road vibration + distortion
                accel = grav_dev + rng.normal(0, 0.7, (WINDOW, 3))
                mag = R.T @ _ned_ref_vec(FIELD_REF_UT+10, INC_REF_DEG-15, DEC_REF_DEG+18) + rng.normal(0, 3.0, (WINDOW, 3))
            elif kind == "walking":                        # rhythmic stepping ~1.8 Hz
                step = 3.0 * np.sin(2*np.pi*1.8*t)
                accel = grav_dev + np.c_[0.6*step, 0.6*step, step] + rng.normal(0, 0.2, (WINDOW, 3))
                mag = (R.T @ B_ned) + rng.normal(0, 1.0, (WINDOW, 3))
            yield {"accel": accel, "gyro": None, "mag": mag, "baro": None,
                   "quat": np.tile(q, (WINDOW, 1)), "label": SYN_TRUE[kind], "kind": kind}

## 5. Synthetic verification
Generate the four constructions (still body, parked-idling vehicle, driving vehicle, walking), compute every feature on each window and apply the rule. This checks that the extractor runs and that the thresholds split the constructed cases as designed. **This is a construction check, not a performance estimate** — the windows are built to be separable, so the numbers below only confirm the math and the threshold placement, not real-world accuracy.

In [ ]:
wins = load_windows("synthetic", n_per=200, seed=0)

rows = []
for w in wins:
    label, _ = locomotion(w)
    mf = magnetic_features(w["mag"], quat=w["quat"])        # compute all features for every window (plotting)
    rows.append({"kind": w["kind"], "true": w["label"], "verdict": label,
                 "movement": movement(w["accel"]), "rhythm": step_rhythm(w["accel"]),
                 "idle": idle_band_ratio(w["accel"]),
                 "field_mag": mf["field_mag"], "field_var": mf["field_var"],
                 "inc_dev": abs(mf["inclination"] - INC_REF_DEG),
                 "dec_dev": ang_diff(mf["declination"], DEC_REF_DEG)})
res = pd.DataFrame(rows)

print("per-kind feature medians:")
print(res.groupby("kind")[["movement", "rhythm", "idle", "field_mag", "field_var", "inc_dev", "dec_dev"]]
        .median().round(2).to_string())
print("\nverdict counts:", res["verdict"].value_counts().to_dict())

In [ ]:
# Where each threshold sits relative to the four constructions
panels = [("rhythm", STEP_T, "step_rhythm (autocorr peak)", False),
          ("idle", IDLE_T, "idle_band_ratio (10-25 Hz)", True),
          ("inc_dev", INC_DEV_T, "|inclination - baseline| (deg)", False),
          ("field_var", FIELD_VAR_T, "field magnitude std (uT)", False)]
order = ["stationary", "vehicle_idle", "vehicle_moving", "walking"]
kind_color = {"stationary": "#55A868", "vehicle_idle": "#DD8452",
              "vehicle_moving": "#C44E52", "walking": "#4C72B0"}

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, (col, thr, title, logx) in zip(axes.ravel(), panels):
    data = [res.loc[res.kind == k, col].to_numpy() for k in order]
    bp = ax.boxplot(data, orientation="horizontal", tick_labels=order, showfliers=False, patch_artist=True)
    for patch, k in zip(bp["boxes"], order):
        patch.set_facecolor(kind_color[k]); patch.set_alpha(0.6)
    ax.axvline(thr, ls="--", color="black", label=f"threshold = {thr:g}")
    if logx:
        ax.set_xscale("log")
    ax.set_xlabel(title); ax.legend(loc="lower right", fontsize=9)
fig.suptitle("Synthetic feature separation — thresholds split the constructed classes", fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
cm = confusion_matrix(res["true"], res["verdict"], labels=CLASSES)
acc = (res["true"] == res["verdict"]).mean()

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(3), CLASSES); ax.set_yticks(range(3), CLASSES)
ax.set_xlabel("predicted locomotion"); ax.set_ylabel("true locomotion (synthetic)")
for i in range(3):
    for j in range(3):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black")
fig.colorbar(im, label="windows")
ax.set_title(f"Synthetic confusion matrix ({acc*100:.0f}% on constructed cases)")
ax.grid(False)
plt.tight_layout(); plt.show()
print("stationary <-> in_vehicle off-diagonal (the target confusion):",
      int(cm[0, 2] + cm[2, 0]), "windows")

## 6. Graceful degradation
The same vehicle window read three ways, to show which features survive as orientation information is removed:

- **full_ned** (quaternion): magnitude, variance, inclination, declination — all cues available.
- **gravity_only** (no quaternion; gravity from the still accelerometer): inclination still recovered as the field's dip from vertical; declination drops to NaN (no true-north reference), so the `dec_dev` cue is skipped.
- **magnitude_only** (no orientation at all): only field magnitude and variance; both angle cues skipped.

Field magnitude, field variance and the engine-idle FFT need no orientation and no geomagnetic baseline, so they anchor the rule even in the weakest mode. The rule still reaches `in_vehicle` in every mode here because the field-magnitude offset and variance alone clear their thresholds.

In [ ]:
demo = next(w for w in load_windows("synthetic", n_per=1, seed=7) if w["kind"] == "vehicle_idle")

modes = {
    "full_ned":       magnetic_features(demo["mag"], quat=demo["quat"]),
    "gravity_only":   magnetic_features(demo["mag"], gravity=demo["accel"].mean(axis=0)),
    "magnitude_only": magnetic_features(demo["mag"]),
}
print("magnetic features by de-rotation mode:")
for name, mf in modes.items():
    inc = f"{mf['inclination']:.1f}" if not np.isnan(mf["inclination"]) else "  NaN"
    dec = f"{mf['declination']:.1f}" if not np.isnan(mf["declination"]) else "  NaN"
    print(f"  {name:14s} field_mag={mf['field_mag']:5.1f}  field_var={mf['field_var']:4.2f}  "
          f"inclination={inc}  declination={dec}")

print("\nrule verdict per available-input mode:")
full = dict(demo)
grav = {k: v for k, v in demo.items() if k != "quat"}            # drop quaternion -> gravity_only path
print("  with quaternion   ->", locomotion(full)[0])
print("  no quaternion     ->", locomotion(grav)[0])

### Findings
- **The stationary <-> in_vehicle fix rests on the magnetometer, de-rotated, plus an engine-idle FFT — GPS-free and deterministic.** Movement alone cannot separate a still body from a stopped or smooth-riding vehicle (the mentor pipeline's ~5000-window leak). De-rotating the field into North-East-Down exposes the vehicle's ferromagnetic distortion as an inclination / declination / magnitude shift away from the geomagnetic baseline, with higher field variance; a narrowband 10-25 Hz tone corroborates a running engine. On the synthetic constructions the two cues are complementary — the parked-idling case is caught by both the field distortion and the idle tone, while the driving case (broadband road vibration, idle ratio ~1) is caught by the field distortion alone.
- **The cues split into baseline-dependent and baseline-free.** Magnitude and angle deviation need the local geomagnetic reference (WMM/IGRF for the site, or a recent stationary-outdoor self-calibration window). Field variance and the engine-idle FFT are absolute thresholds and need no reference, so they still flag a vehicle when the location/baseline is unknown.
- **Graceful degradation is built in.** With a quaternion the rule uses inclination + declination; with gravity only it keeps inclination and skips declination; with neither it falls back to field magnitude + variance + idle FFT. Each missing input drops its cue rather than breaking the rule.
- **on_foot is unchanged and gated.** Rhythm (autocorrelation peak at cadence) handles stepping as before; a `MOVE_FLOOR` gate stops a strong periodic engine tone from being misread as walking, since a pure tone also raises the autocorrelation peak.
- **Thresholds are PROVISIONAL.** Every constant was placed to separate the synthetic constructions, which are built to be separable — so the synthetic confusion confirms only that the features compute and the thresholds sit between the classes, not any real accuracy. No real-data validation and no accuracy claims this loop.
- **Next loop: real data.** Plug SHL (Hips/pocket, 100 Hz, with magnetometer + pressure + quaternion) and an own-phone CSV into `load_windows`, refit the baseline and thresholds on real stationary-vs-vehicle windows, and report the actual stationary <-> in_vehicle confusion against the mentor pipeline. Open items for that pass: barometer altitude-variance as a moving-vs-stopped cue, self-calibration of the geomagnetic baseline, and behaviour on trains/subways where the chassis distortion and idle tone differ from road vehicles.

## 7. Real-data check on PAMAP2 (loop 2)
The synthetic check above only confirmed the math. This loop runs the rule on real inertial data — PAMAP2, **chest** IMU (the chest sits still on the torso in a car, so it senses the vehicle, not the hand). Read the scope honestly before any number:

- **Single-subject DEMO, not a validation.** Only one PAMAP2 subject recorded car driving (Section 7.1), so `in_vehicle` cannot be split subject-independently and nothing here generalizes across cars or sensors.
- **Moving-driving only.** PAMAP2 driving is on-road motion, not parked-idle, so this tests the *moving-vehicle* magnetic signature, not the engine-idle case the synthetic `vehicle_idle` covered.
- **Degraded rule.** PAMAP2 has no barometer and its orientation columns are flagged invalid in the dataset readme, so there is **no quaternion**. De-rotation runs in **gravity-only** mode: inclination is recovered from the accelerometer gravity vector, declination is NaN, so the `dec_dev` cue is inactive. Active cues: `mag_dev`, `inc_dev`, `field_var`, `idle` (100 Hz, so the idle band is within Nyquist).
- Chest columns: accel +-16g (21-23), magnetometer in uT (30-32). 100 Hz, 5 s windows, 50% overlap. Classes folded to `stationary` (lying/sitting/standing), `on_foot` (walking/running/nordic), `in_vehicle` (driving, id 11); cycling/stairs/household excluded.

### 7.1 STEP 0 — scan for driving before trusting any accuracy
Count 5 s windows of activity 11 (car driving) per subject across Protocol + Optional. This decides validation vs demo.

In [ ]:
import glob, re

PAMAP2_BASE = "/home/voare/Documents/Synheart/Kinematics/Dataset/PAMAP2_data/PAMAP2_Dataset"
PAMAP2_FILES = sorted(glob.glob(PAMAP2_BASE + "/Protocol/*.dat")) + sorted(glob.glob(PAMAP2_BASE + "/Optional/*.dat"))
STEP_OVERLAP = WINDOW // 2                          # 50% overlap -> 250-sample hop

CHEST_ACC = [21, 22, 23]                            # chest accelerometer +-16g (m/s^2)
CHEST_MAG = [30, 31, 32]                            # chest magnetometer (uT)
ACT_NAME = {1: "lying", 2: "sitting", 3: "standing", 4: "walking", 5: "running",
            7: "nordic_walk", 11: "driving"}
ACT_GROUP = {"lying": "stationary", "sitting": "stationary", "standing": "stationary",
             "walking": "on_foot", "running": "on_foot", "nordic_walk": "on_foot",
             "driving": "in_vehicle"}

def _runs(act):
    change = np.where(np.diff(act) != 0)[0] + 1
    return zip(np.r_[0, change], np.r_[change, len(act)])

driving_per_subject = {}
for path in PAMAP2_FILES:
    subj = int(re.search(r"subject(\d+)", path).group(1))
    act = pd.read_csv(path, sep=r"\s+", header=None, usecols=[1])[1].to_numpy()
    n = sum(len(range(s0, e0 - WINDOW + 1, STEP_OVERLAP)) for s0, e0 in _runs(act) if act[s0] == 11)
    driving_per_subject[subj] = driving_per_subject.get(subj, 0) + n

print("activity-11 (car driving) 5 s / 50%-overlap windows per subject:")
for s in sorted(driving_per_subject):
    print(f"  subject {s}: {driving_per_subject[s]:4d}" + ("  <-- has driving" if driving_per_subject[s] else ""))
drivers = [s for s, n in driving_per_subject.items() if n]
print(f"\nsubjects with driving: {drivers}  (n={len(drivers)})")
print("=> SINGLE-SUBJECT DEMO" if len(drivers) <= 2 else "=> multi-subject validation")
DEMO_SUBJECT = drivers[0]

### 7.2 Load the demo subject and extract features
Slide windows within each activity run on the demo subject's chest stream, skip windows with gaps, and run the frozen feature extractor. No quaternion is passed, so `magnetic_features` takes the gravity-only branch (inclination yes, declination NaN).

In [ ]:
def load_pamap2_chest(subject):
    """Windows for one PAMAP2 subject as dict(accel, mag, quat=None, label, kind). Gravity-only de-rotation."""
    out = []
    for path in [p for p in PAMAP2_FILES if f"subject{subject}" in p]:
        raw = pd.read_csv(path, sep=r"\s+", header=None, usecols=[1] + CHEST_ACC + CHEST_MAG)
        act = raw[1].to_numpy(); acc = raw[CHEST_ACC].to_numpy(); mag = raw[CHEST_MAG].to_numpy()
        for s0, e0 in _runs(act):
            a = act[s0]
            if a not in ACT_NAME:
                continue
            for s in range(s0, e0 - WINDOW + 1, STEP_OVERLAP):
                wa, wm = acc[s:s + WINDOW], mag[s:s + WINDOW]
                if np.isnan(wa).any() or np.isnan(wm).any():
                    continue
                out.append({"accel": wa, "mag": wm, "quat": None,
                            "label": ACT_GROUP[ACT_NAME[a]], "kind": ACT_NAME[a]})
    return out

pw = load_pamap2_chest(DEMO_SUBJECT)
prows = []
for w in pw:
    mf = magnetic_features(w["mag"], gravity=w["accel"].mean(axis=0))     # gravity-only branch
    prows.append({"kind": w["kind"], "true": w["label"],
                  "movement": movement(w["accel"]), "rhythm": step_rhythm(w["accel"]),
                  "idle": idle_band_ratio(w["accel"]), "field_mag": mf["field_mag"],
                  "field_var": mf["field_var"], "inclination": mf["inclination"],
                  "mode": mf["mode"]})
pres = pd.DataFrame(prows)

print(f"DEMO subject {DEMO_SUBJECT} | de-rotation mode: {pres['mode'].iloc[0]} "
      f"(declination NaN -> dec_dev cue inactive)")
print("windows per class:", pres["true"].value_counts().to_dict())
print("\nfeature medians by class:")
print(pres.groupby("true")[["movement", "rhythm", "idle", "field_mag", "field_var", "inclination"]]
        .median().round(2).to_string())

### 7.3 Does the magnetometer separate still-body from in-vehicle?
The loop's core question. Field magnitude, gravity-only inclination and field variance for `stationary` vs `in_vehicle`, side by side. Field variance is shown too, but note it carries body motion and so is confounded; the idle FFT is reported in 7.4 but is a parked-engine cue and is not expected to fire on moving driving.

In [ ]:
pair = pres[pres.true.isin(["stationary", "in_vehicle"])]
feats = [("field_mag", "field magnitude (uT)"), ("inclination", "gravity-only inclination (deg)"),
         ("field_var", "field magnitude std (uT)")]
col = {"stationary": "#55A868", "in_vehicle": "#DD8452"}

fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
for ax, (f, title) in zip(axes, feats):
    data = [pair.loc[pair.true == g, f].to_numpy() for g in ["stationary", "in_vehicle"]]
    bp = ax.boxplot(data, tick_labels=["stationary", "in_vehicle"], showfliers=False, patch_artist=True)
    for patch, g in zip(bp["boxes"], ["stationary", "in_vehicle"]):
        patch.set_facecolor(col[g]); patch.set_alpha(0.6)
    ax.set_ylabel(title)
fig.suptitle(f"PAMAP2 subject {DEMO_SUBJECT}: chest-magnetometer features, stationary vs in_vehicle", fontsize=12)
plt.tight_layout(); plt.show()

s_fm = pair.loc[pair.true == "stationary", "field_mag"]
v_fm = pair.loc[pair.true == "in_vehicle", "field_mag"]
print(f"field magnitude  stationary [{s_fm.min():.1f}, {s_fm.max():.1f}] uT   "
      f"in_vehicle [{v_fm.min():.1f}, {v_fm.max():.1f}] uT")
gap = s_fm.min() - v_fm.max()
print(f"  -> clean gap of {gap:.1f} uT between the two classes" if gap > 0
      else "  -> classes overlap in field magnitude")

### 7.4 Frozen rule on real data — and the baseline it needs
Run the loop-1 `locomotion()` rule unchanged, two ways. First with its synthetic config baseline (48 uT / 62 deg); then with a **self-calibrated** baseline taken from the demo subject's own stationary windows (the deployment option documented in loop 1). The synthetic baseline is the wrong magnitude for this sensor, so the magnitude/angle cues misfire — this is the loop-1 "baseline-dependent cues" caveat made concrete.

In [ ]:
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support

def report(true, pred, title):
    cm = confusion_matrix(true, pred, labels=CLASSES)
    acc = (np.array(true) == np.array(pred)).mean()
    P, R, F, _ = precision_recall_fscore_support(true, pred, labels=CLASSES, zero_division=0)
    print(f"\n{title}   (overall {acc*100:.1f}%)")
    print("  rows=true cols=pred  " + " ".join(f"{c[:10]:>10s}" for c in CLASSES))
    for i, c in enumerate(CLASSES):
        print(f"  {c:11s} " + " ".join(f"{cm[i,j]:10d}" for j in range(3)) +
              f"   P={P[i]:.2f} R={R[i]:.2f} F1={F[i]:.2f}")
    si, vi = CLASSES.index("stationary"), CLASSES.index("in_vehicle")
    print(f"  stationary<->in_vehicle off-diagonal: {cm[si,vi]+cm[vi,si]} "
          f"(stat->veh {cm[si,vi]}, veh->stat {cm[vi,si]})  <-- the target confusion")
    return cm

# self-calibrated baseline = this subject's stationary medians
stat = pres[pres.true == "stationary"]
SELF_REF = {"field_mag": float(stat.field_mag.median()),
            "inclination": float(stat.inclination.median()),
            "declination": DEC_REF_DEG}
print(f"self-cal baseline: field_mag={SELF_REF['field_mag']:.1f} uT, inclination={SELF_REF['inclination']:.1f} deg")

pred_syn = [locomotion(w)[0] for w in pw]                         # ref=None -> synthetic config baseline
pred_self = [locomotion(w, ref=SELF_REF)[0] for w in pw]         # self-calibrated baseline, frozen thresholds
report(pres.true, pred_syn, "A) frozen thresholds, SYNTHETIC baseline (48 uT / 62 deg)")
report(pres.true, pred_self, "B) frozen thresholds, SELF-CAL baseline")

### 7.5 Re-derive the thresholds on real data
The synthetic thresholds were placed on constructed windows; like every other axis, they shift on real data. From the demo subject's distributions: `mag_dev` widens 8 -> 14 uT (stationary deviation reaches ~9 uT, in_vehicle starts at ~18 uT); `inc_dev` widens 10 -> 25 deg (gravity-only dip is noisy without a quaternion, so stationary inclination spreads more); `field_var` and `idle` are **disabled** (thresholds set above the observed range) because on moving driving the field variance is dominated by body motion and the idle FFT only fires on a parked engine. The decision then rests on the magnetometer magnitude/inclination deviation, exactly as designed. These constants are **fitted on one subject** — a demo baseline, not a deployable threshold.

In [ ]:
def classify(win, ref, T):
    """Same branching as loop-1 locomotion(), with thresholds exposed for the re-derivation study."""
    accel, mag = win["accel"], win["mag"]
    if step_rhythm(accel) > T["STEP_T"] and movement(accel) > T["MOVE_FLOOR"]:
        return "on_foot"
    mf = magnetic_features(mag, quat=win.get("quat"), gravity=accel.mean(axis=0))
    cues = [abs(mf["field_mag"] - ref["field_mag"]) > T["MAG_DEV_T"],
            (not np.isnan(mf["inclination"])) and abs(mf["inclination"] - ref["inclination"]) > T["INC_DEV_T"],
            (not np.isnan(mf["declination"])) and ang_diff(mf["declination"], ref["declination"]) > T["DEC_DEV_T"],
            mf["field_var"] > T["FIELD_VAR_T"],
            (not np.isnan(idle_band_ratio(accel))) and idle_band_ratio(accel) > T["IDLE_T"]]
    return "in_vehicle" if any(cues) else "stationary"

FROZEN_T = dict(STEP_T=STEP_T, MOVE_FLOOR=MOVE_FLOOR, MAG_DEV_T=MAG_DEV_T, INC_DEV_T=INC_DEV_T,
                DEC_DEV_T=DEC_DEV_T, FIELD_VAR_T=FIELD_VAR_T, IDLE_T=IDLE_T)
REDERIVED_T = {**FROZEN_T, "MAG_DEV_T": 14.0, "INC_DEV_T": 25.0, "FIELD_VAR_T": 1e9, "IDLE_T": 1e9}

print("threshold changes (synthetic -> re-derived on PAMAP2 demo):")
for k in ["MAG_DEV_T", "INC_DEV_T", "FIELD_VAR_T", "IDLE_T"]:
    a, b = FROZEN_T[k], REDERIVED_T[k]
    print(f"  {k:12s} {a:>6.1f} -> {('OFF' if b>=1e9 else f'{b:.1f}'):>6s}")

pred_rd = [classify(w, SELF_REF, REDERIVED_T) for w in pw]
cm = report(pres.true, pred_rd, "\nC) RE-DERIVED thresholds, self-cal baseline")

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(3), CLASSES); ax.set_yticks(range(3), CLASSES)
ax.set_xlabel("predicted locomotion"); ax.set_ylabel("true locomotion (PAMAP2 subject %d)" % DEMO_SUBJECT)
for i in range(3):
    for j in range(3):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black")
fig.colorbar(im, label="windows")
ax.set_title("PAMAP2 demo — re-derived rule, self-cal baseline")
ax.grid(False)
plt.tight_layout(); plt.show()

### Findings — loop 2 (PAMAP2 real data)
- **Scope: single-subject DEMO.** Only PAMAP2 subject 101 recorded car driving (217 raw / 189 gap-free windows); the other eight subjects have none. So `in_vehicle` cannot be split subject-independently and nothing here generalizes across cars or sensors. It is also **moving-driving**, not parked-idle, and a **degraded** rule (no quaternion -> gravity-only de-rotation, declination inactive; no barometer).
- **The magnetometer separates still-body from in-vehicle, clearly.** On the chest, field magnitude alone has a clean gap — stationary ~59-77 uT vs in_vehicle ~3-50 uT (median 68 -> 36 uT): the steel cabin attenuates and distorts the geomagnetic field. Gravity-only inclination corroborates (stationary ~56 deg -> in_vehicle ~-4 deg). This is the loop's question, and on this subject the answer is yes.
- **The cue needs the right baseline.** With the synthetic baseline (48 uT) the frozen rule calls every stationary window in_vehicle, because the magnitude deviation is measured against the wrong reference. Self-calibration from the subject's own stationary windows fixes it. This is exactly the baseline-dependence flagged in loop 1, now demonstrated on real data.
- **Thresholds shifted on real data, as expected.** `mag_dev` 8 -> 14 uT and `inc_dev` 10 -> 25 deg (gravity-only dip is noisier without a quaternion); `field_var` and `idle` were disabled — field variance is dominated by body motion here, and the idle FFT fires only on a parked engine, while PAMAP2 driving is moving. With these, overall accuracy is ~88% and the **stationary <-> in_vehicle confusion is 0 windows** — the leak this axis set out to fix.
- **Residual error is the on_foot gate, not the magnetic split.** The misses are ~46 driving windows read as on_foot (in_vehicle recall ~0.76): moving-car vibration trips the rhythm gate (rhythm > 0.30 and movement > 0.30). These are driving -> on_foot, never driving -> stationary, so the magnetic separation is intact; the fix is a tighter cadence/movement gate, deferred.
- **Next: multiple drivers and orientation.** A real validation needs several subjects and cars (SHL Hips/pocket, which also brings a valid quaternion for full North-East-Down de-rotation, a barometer, and parked-idle segments to exercise the idle FFT). The thresholds re-derived here are a one-subject demo baseline, not deployable as-is.